## 01 Data Gathering

#### Purpose
- Gather the data and information in relation to the flood insurance program of US
- Download is done via application program interface (API)

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III. 

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-07

### Section 0. Load the libraries and working directory

In [1]:
# Import tools
import numpy as np
import pandas as pd
import scipy as sp
import os
from pathlib import Path
import re
import urllib.request
import os, json, urllib.parse
from dotenv import load_dotenv
import duckdb

# Set working directory
# Repo root = parent of notebooks/ (works on any machine)
working_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_folder = working_directory /"data" / "raw"
processed_data_folder = working_directory /"data" /"processed"
reports_folder = working_directory / "reports"

# API Key
# API key (stored in .env at the repo root; never written in the notebook)
load_dotenv(working_directory / ".env")
KEY = os.environ.get("CENSUS_API_KEY")
assert KEY, "CENSUS_API_KEY not found in .env"

### Section 1. Gather the datasets via API

Our datasets include the following (all downloaded in full, unchanged, into `data/raw/`):
1. NFIP Redacted Policies v3: about 74.7M policy terms (OpenFEMA bulk Parquet)
2. NFIP Redacted Claims v3: about 2.7M claims (OpenFEMA bulk Parquet)
3. Disaster Declarations Summaries (OpenFEMA)
4. NFIP Community Status Book (OpenFEMA)
5. FRED macroeconomic series
6. Census ACS 5-year (2022), tract level

The two NFIP bulk files are large, so they are downloaded with `curl`, which can **resume** an interrupted download. Python's `urllib` is refused by FEMA's server for these files (HTTP 403).

In [2]:
# Create a function to download the datasets
RAW = raw_data_folder
RAW.mkdir(parents=True, exist_ok=True)

def download(url, dest, headers=None):
    """Save a file once; skip it if it's already there.
    Writes to a .part file first, so an interrupted download is never mistaken for a complete one."""
    dest = RAW / dest
    if dest.exists():
        print(f"skip (exists): {dest.relative_to(RAW)}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    print(f"downloading {dest.relative_to(RAW)} ...")
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=600) as r, open(part, "wb") as f:
        done = 0
        while chunk := r.read(8 * 1024 * 1024):
            f.write(chunk)
            done += len(chunk)
            if done % (512 * 1024 * 1024) < len(chunk):      # progress about every 512 MB
                print(f"  {done / 1e9:.1f} GB")
    part.rename(dest)                                        # only reached if the download finished
    print(f"saved {dest.relative_to(RAW)} ({dest.stat().st_size / 1e6:,.1f} MB)")
    return dest

In [3]:
# Download the full NFIP bulk files with curl (resumable; re-run this cell to continue an interrupted download)
import shutil
import subprocess

def download_bulk(url, dest):
    """Download a large file with curl into a .part file, then rename it when complete."""
    dest = RAW / dest
    if dest.exists():
        print(f"skip (exists): {dest.relative_to(RAW)}")
        return dest
    assert shutil.which("curl"), "curl not found"
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    print(f"downloading {dest.relative_to(RAW)} (progress is shown in the terminal running Jupyter) ...")
    subprocess.run(["curl", "-L", "--fail", "--retry", "5", "-C", "-", "-o", str(part), url], check=True)
    part.rename(dest)                                        # only reached if curl finished successfully
    print(f"saved {dest.relative_to(RAW)} ({dest.stat().st_size / 1e9:,.2f} GB)")
    return dest

nfip_bulk = {
    "fema/nfip_claims_v3.parquet":
        "https://www.fema.gov/about/reports-and-data/openfema/v3/NfipClaimsV3.parquet",
    "fema/nfip_policies_v3.parquet":                        # the large one; may take an hour or more
        "https://www.fema.gov/about/reports-and-data/openfema/v3/NfipPoliciesV3.parquet",
}
for dest, url in nfip_bulk.items():
    download_bulk(url, dest)

downloading fema/nfip_claims_v3.parquet (progress is shown in the terminal running Jupyter) ...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  156M  100  156M    0     0  23.5M      0  0:00:06  0:00:06 --:--:-- 25.6M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

saved fema/nfip_claims_v3.parquet (0.16 GB)
downloading fema/nfip_policies_v3.parquet (progress is shown in the terminal running Jupyter) ...


 99 3541M   99 3528M    0     0  39.9M      0  0:01:28  0:01:28 --:--:-- 39.7M

saved fema/nfip_policies_v3.parquet (3.71 GB)


100 3541M  100 3541M    0     0  39.9M      0  0:01:28  0:01:28 --:--:-- 42.0M


In [4]:
# Download the small OpenFEMA files
fema = {
    "fema/disaster_declarations_summaries_v2.parquet":
        "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries.parquet",
    "fema/nfip_community_status_book_v1.parquet":
        "https://www.fema.gov/api/open/v1/NfipCommunityStatusBook.parquet",
}
for dest, url in fema.items():
    download(url, dest)

downloading fema/disaster_declarations_summaries_v2.parquet ...
saved fema/disaster_declarations_summaries_v2.parquet (7.4 MB)
downloading fema/nfip_community_status_book_v1.parquet ...
saved fema/nfip_community_status_book_v1.parquet (1.0 MB)


In [5]:
# Download the FRED macroeconomic series
for sid in ["MORTGAGE30US", "DGS10", "UNRATE", "CPIAUCSL", "CSUSHPINSA", "FEDFUNDS"]:
    download(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}", f"fred/{sid}.csv")

downloading fred/MORTGAGE30US.csv ...
saved fred/MORTGAGE30US.csv (0.0 MB)
downloading fred/DGS10.csv ...
saved fred/DGS10.csv (0.3 MB)
downloading fred/UNRATE.csv ...
saved fred/UNRATE.csv (0.0 MB)
downloading fred/CPIAUCSL.csv ...
saved fred/CPIAUCSL.csv (0.0 MB)
downloading fred/CSUSHPINSA.csv ...
saved fred/CSUSHPINSA.csv (0.0 MB)
downloading fred/FEDFUNDS.csv ...
saved fred/FEDFUNDS.csv (0.0 MB)


In [6]:
# Download Census ACS 5-year (2022) tract-level estimates
VARS = {"B19013_001E": "median_household_income", "B25077_001E": "median_home_value",
        "B01002_001E": "median_age", "B25003_001E": "occupied_housing_units",
        "B25003_002E": "owner_occupied_units", "B01003_001E": "total_population"}
STATES = ["12"]   # Florida; add more FIPS codes as needed

dest = RAW / "census" / "acs5_2022_tract.csv"
if dest.exists():
    print(f"skip (exists): {dest.relative_to(RAW)}")
else:
    frames = []
    for st in STATES:
        q = urllib.parse.urlencode({"get": "NAME," + ",".join(VARS), "for": "tract:*",
                                    "in": f"state:{st}", "key": KEY})
        with urllib.request.urlopen(f"https://api.census.gov/data/2022/acs/acs5?{q}") as r:
            body = r.read().decode()
            final_url = r.geturl()
        try:
            data = json.loads(body)
        except json.JSONDecodeError:
            raise RuntimeError(f"State {st}: Census returned a non-JSON page ({final_url}). "
                               "Check that the key is loaded and activated.")
        frames.append(pd.DataFrame(data[1:], columns=data[0]))
        print(f"state {st}: {len(data) - 1:,} tracts")
    dest.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(frames).rename(columns=VARS).to_csv(dest, index=False)   # raw values, sentinels kept
    print(f"saved {dest.relative_to(RAW)}")

state 12: 5,160 tracts
saved census/acs5_2022_tract.csv


In [7]:
# Verify downloads: file sizes and row counts
import pyarrow.parquet as pq

for f in sorted(RAW.rglob("*")):
    if f.is_file() and f.name != ".gitkeep":
        size = f"{f.stat().st_size / 1e6:,.1f} MB"
        rows = f"{pq.ParquetFile(f).metadata.num_rows:,} rows" if f.suffix == ".parquet" else ""
        flag = "  ⚠ incomplete download, delete and re-run" if f.suffix == ".part" else ""
        print(f"{str(f.relative_to(RAW)):55s} {size:>12s}  {rows}{flag}")
# Expected: claims ≈ 2.7M rows, policies ≈ 74.7M rows

.DS_Store                                                     0.0 MB  
census/acs5_2022_tract.csv                                    0.5 MB  
fema/disaster_declarations_summaries_v2.parquet               7.4 MB  70,599 rows
fema/nfip_claims_v3.parquet                                 163.8 MB  2,725,989 rows
fema/nfip_community_status_book_v1.parquet                    1.0 MB  25,133 rows
fema/nfip_policies_v3.parquet                             3,713.0 MB  74,739,461 rows
fred/CPIAUCSL.csv                                             0.0 MB  
fred/CSUSHPINSA.csv                                           0.0 MB  
fred/DGS10.csv                                                0.3 MB  
fred/FEDFUNDS.csv                                             0.0 MB  
fred/MORTGAGE30US.csv                                         0.0 MB  
fred/UNRATE.csv                                               0.0 MB  


#### Loading the full files
The policies file (about 74.7M rows × 87 columns) is too large for pandas in memory. Query it with DuckDB directly from disk, and bring into pandas only the rows and columns you need.

In [8]:
# Overview of the full NFIP files, computed on disk (nothing large is loaded into memory)
policies_path = RAW / "fema" / "nfip_policies_v3.parquet"
claims_path = RAW / "fema" / "nfip_claims_v3.parquet"

print(duckdb.sql(f"""
    SELECT 'policies' AS dataset, COUNT(*) AS n_rows,
           MIN(policyEffectiveDate) AS first_date, MAX(policyEffectiveDate) AS last_date
    FROM '{policies_path}'
    UNION ALL
    SELECT 'claims', COUNT(*), MIN(dateOfLoss), MAX(dateOfLoss)
    FROM '{claims_path}'
"""))

# Rows per state (top 10)
print(duckdb.sql(f"""
    SELECT propertyState, COUNT(*) AS n_policy_terms
    FROM '{policies_path}'
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
"""))

┌──────────┬──────────┬────────────┬────────────┐
│ dataset  │  n_rows  │ first_date │ last_date  │
│ varchar  │  int64   │    date    │    date    │
├──────────┼──────────┼────────────┼────────────┤
│ policies │ 74739461 │ 2009-01-01 │ 2027-08-05 │
│ claims   │  2725989 │ 1978-01-01 │ 2026-09-07 │
└──────────┴──────────┴────────────┴────────────┘

┌───────────────┬────────────────┐
│ propertyState │ n_policy_terms │
│    varchar    │     int64      │
├───────────────┼────────────────┤
│ FL            │       22377861 │
│ TX            │       11612805 │
│ LA            │        8456525 │
│ CA            │        3953578 │
│ NJ            │        2918984 │
│ SC            │        2715811 │
│ NY            │        2705125 │
│ NC            │        2207699 │
│ VA            │        1668793 │
│ GA            │        1461337 │
└───────────────┴────────────────┘
  10 rows              2 columns



In [9]:
# Example: bring one slice into pandas (Florida, 2019-2023)
policies_fl = duckdb.sql(f"""
    SELECT *
    FROM '{policies_path}'
    WHERE propertyState = 'FL'
      AND policyEffectiveDate >= '2019-01-01'
      AND policyEffectiveDate <  '2024-01-01'
""").df()
policies_fl.shape

(5681104, 87)